# Segmasker Low-Complexity Detection

Protein sequences sometimes contain regions of unusually low compositional complexity, such as poly-alanine stretches, proline-rich tracts, or runs of charged residues. These regions are often intrinsically disordered and can cause false positives in homology searches because their biased composition inflates alignment scores. Detecting and masking low-complexity regions is a standard preprocessing step in sequence analysis pipelines, and it is equally important when evaluating designed protein sequences.

Segmasker is NCBI's implementation of the SEG algorithm, which identifies low-complexity regions by computing local compositional complexity within a sliding window. This notebook demonstrates how to use Segmasker to scan protein sequences and quantify their low-complexity content, comparing a natural well-folded protein against an artificial sequence containing a poly-alanine insertion.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("segmasker")
display_overview("segmasker")
display_docs_section("segmasker", "Background")

# Segmasker

> Segmasker detects [low-complexity regions](https://en.wikipedia.org/wiki/Low_complexity_regions_in_proteins) in protein sequences using NCBI's SEG algorithm. Low-complexity regions are stretches of amino acids with biased composition (e.g., polyalanine runs, proline-rich regions, or glutamine repeats) that can cause spurious hits in homology searches and may indicate disordered or non-globular regions.

## Background

Low-complexity regions (LCRs) are protein segments with reduced amino acid diversity compared to typical globular proteins. The SEG algorithm (Wootton & Federhen, 1993) identifies these regions by computing local compositional complexity using [Shannon entropy](https://en.wikipedia.org/wiki/Entropy_\(information_theory\)) within sliding windows.

LCRs are common in eukaryotic proteomes (up to 20-25% of residues) and are associated with:

* **[Intrinsically disordered regions](https://en.wikipedia.org/wiki/Intrinsically_disordered_proteins)** that lack stable 3D structure
* **Repeat expansions** linked to neurodegenerative diseases (e.g., polyglutamine in [Huntington's](https://en.wikipedia.org/wiki/Huntington%27s_disease))
* **Compositionally biased linkers** between structured domains
* **False positives in BLAST** -- LCRs match other LCRs regardless of evolutionary relationship

In protein design, high low-complexity content often signals poor sequence quality, since natural globular proteins typically have low-complexity fractions below 0.1.

## Available tools

In [2]:
display_available_tools("segmasker")

- **`run_segmasker()`** — Detect low-complexity regions in protein sequences using NCBI segmasker

### `run_segmasker`

Segmasker scans one or more protein sequences using a sliding-window complexity analysis and returns the fraction and count of residues classified as low-complexity for each sequence. The algorithm parameters — window size, low-complexity threshold (`locut`), and high-complexity threshold (`hicut`) — can be tuned to adjust sensitivity. This example compares the N-terminal region of human hemoglobin alpha (a well-folded globular protein with no compositional bias) against a synthetic variant with a poly-alanine insertion, demonstrating that Segmasker reliably detects the low-complexity stretch while leaving the natural sequence unmasked.

In [3]:
from proto_tools import SegmaskerInput, SegmaskerConfig, run_segmasker

In [4]:
# Display input docs
display_api_reference("segmasker", "input", "run_segmasker")

# Compare a well-folded protein vs one with a low-complexity stretch
sequences = [
    "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH",  # Hemoglobin alpha (well-folded)
    "MVLSPADKAAAAAAAAAAAAAAAAAAAAAAGAEALERMFLSFPTTKTYFPHF",  # Artificial poly-alanine insert
]
inputs = SegmaskerInput(sequences=sequences)

**Input** — `SegmaskerInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequences` | `List[string]` | required | Protein sequence(s) to analyze for low-complexity regions. Can be provided as: |

In [5]:
# Display config docs
display_api_reference("segmasker", "config", "run_segmasker")

# Default config: window=15, locut=1.8, hicut=3.4
config = SegmaskerConfig()

**Config** — `SegmaskerConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `window` | `integer` | `15` | Window size for analyzing sequence complexity. The algorithm examines the sequence in sliding windows of this size. Larger windows are less sensitive to short low-complexity regions. Typical range: 12-20. Must be at least 1. Default: 15. |
| `locut` | `number` | `1.8` | Low-complexity cutoff threshold. Regions with complexity scores below this threshold are classified as low-complexity. Lower values identify only the most extreme low-complexity regions. Typical |
| `hicut` | `number` | `3.4` | High-complexity cutoff threshold. Regions with complexity scores above this threshold are classified as high-complexity. Used to define the transition between masked and unmasked regions. Should be greater than `locut`. Typical range: 2.5-3.8. Default: 3.4. |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cpu` | Device to run the tool on. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [6]:
# Run the segmasker tool with default configuration
result = run_segmasker(inputs, config)

Running run_segmasker [00:00]

In [7]:
display_api_reference("segmasker", "output", "run_segmasker")

labels = ["Hemoglobin alpha", "Poly-Ala insert"]
for label, frac, count, length in zip(
    labels, result.low_complexity_fractions, result.low_complexity_counts, result.sequence_lengths
):
    print(f"{label}: {frac:.1%} low-complexity ({count}/{length} residues)")

**Output** — `SegmaskerOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `low_complexity_fractions` | `List[number]` | required | Fraction of each sequence classified as low-complexity. Range: 0.0-1.0 where: |
| `low_complexity_counts` | `List[integer]` | required | Number of positions classified as low-complexity in each sequence. Equals `low_complexity_fraction * length`. |
| `sequence_lengths` | `List[integer]` | required | Length of each input sequence in amino acids. |

Hemoglobin alpha: 0.0% low-complexity (0/51 residues)
Poly-Ala insert: 50.0% low-complexity (26/52 residues)


In [8]:
# Use a stricter (lower) locut threshold to detect only the most extreme compositional bias
strict_config = SegmaskerConfig(locut=1.4, hicut=3.0)
strict_result = run_segmasker(inputs, strict_config)

for label, frac, count, length in zip(
    labels, strict_result.low_complexity_fractions, strict_result.low_complexity_counts, strict_result.sequence_lengths
):
    print(f"{label} (strict): {frac:.1%} low-complexity ({count}/{length} residues)")

Running run_segmasker [00:00]

Hemoglobin alpha (strict): 0.0% low-complexity (0/51 residues)
Poly-Ala insert (strict): 50.0% low-complexity (26/52 residues)


## Export Results

Segmasker results can be exported to CSV or JSON for downstream filtering. The exported table includes the sequence length, low-complexity residue count, and low-complexity fraction for each input sequence.

In [9]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="segmasker_results", export_path=output_dir, file_format="csv")
print(f"Segmasker results exported to {output_dir}/segmasker_results.csv")

Segmasker results exported to example_output/segmasker_results.csv
